# Day 02 — Conditions and Loops
**Module 01: Python + Async + FastAPI for LLM Engineering**

> **What you'll learn:** if/elif/else · comparison operators · boolean operators · `in` operator · for loops · enumerate · accumulation pattern · while loops · exponential backoff · break · continue · list comprehension · dict comprehension

> **Connection to Module 00:** The routing rules you wrote by hand in `## Task` — *"Route TRACK_ORDER queries to logistics"* — become `if/elif/else` in Python today.


---


## 1 — if / elif / else

A **condition** is a question your code asks at runtime. The answer is always `True` or `False`.

```python
if condition:          # runs when True
    ...
elif other_condition:  # runs when first was False, this is True
    ...
else:                  # runs when nothing above matched — always write this
    ...
```

**Three rules:**
1. Colon `:` is required at the end of every `if` / `elif` / `else` line
2. Python checks **top to bottom** and **stops at the first match**
3. Always write `else` — real data always surprises you


In [ ]:
# Simple — grade from a score
score = 72

if score >= 90:
    grade = "A"
elif score >= 80:
    grade = "B"
elif score >= 70:
    grade = "C"
else:
    grade = "F"

print(f"score={score} → grade={grade}")


### Comparison and Boolean Operators

| Operator | Meaning | Example |
|----------|---------|--------|
| `==` | equal to | `status == "Delivered"` |
| `!=` | not equal | `status != "Cancelled"` |
| `>` | greater than | `rating > 3` |
| `<` | less than | `price < 100.0` |
| `>=` | greater or equal | `rating >= 4` |
| `<=` | less or equal | `stock <= 0` |
| `in` | substring / membership | `"cancel" in query.lower()` |
| `and` | both must be True | `is_in_stock and is_recommended` |
| `or` | at least one True | `"cancel" in q or "refund" in q` |
| `not` | flip True↔False | `not is_cancelled` |


In [4]:
price  = 205.21
stock  = 0
rating = 3

is_in_stock    = stock > 0
is_affordable  = price < 300.0
is_recommended = rating >= 4

print(type(is_in_stock), type(is_affordable), type(is_recommended))
print(is_in_stock, is_affordable, is_recommended)

if is_in_stock and is_recommended:
    recommendation = "Great choice — in stock and highly rated!"
elif is_in_stock and not is_recommended:
    recommendation = "In stock but reviews are mixed."
else:
    recommendation = "Out of stock."

print(recommendation)


<class 'bool'> <class 'bool'> <class 'bool'>
False True False
Out of stock.


### ShopSmart — Query Routing

In Module 00 you wrote: *"Route TRACK_ORDER queries to logistics."*  
Here those same rules become Python `if/elif/else`.

**Key pattern:** always call `.lower()` once at the top so matching is case-insensitive.  
Put the **most urgent condition first** — cancel/refund before general tracking.


In [5]:
customer_query = "Where is my order #3042?"
q = customer_query.lower()   # normalise once, use everywhere

if "cancel" in q or "refund" in q:
    agent, urgency = "human_agent", "high"
elif "track" in q or "where" in q or "delivery" in q:
    agent, urgency = "order_agent", "normal"
elif "return" in q or "exchange" in q:
    agent, urgency = "returns_agent", "normal"
elif "price" in q or "discount" in q:
    agent, urgency = "promotions_agent", "normal"
elif "product" in q or "spec" in q:
    agent, urgency = "catalog_agent", "low"
else:
    agent, urgency = "general_agent", "normal"

print(f"Query  : {customer_query}")
print(f"Agent  : {agent}")
print(f"Urgency: {urgency}")


Query  : Where is my order #3042?
Agent  : order_agent
Urgency: normal


In [6]:
# Map database status values → customer-facing messages
# The else handles any unexpected value — real data is always messy
statuses = ["Delivered", "In Transit", "Cancelled", "Refunded", "Unknown"]

for status in statuses:
    if status == "Delivered":
        msg = "Your order has been delivered."
    elif status == "In Transit":
        msg = "Your order is on its way."
    elif status == "Processing":
        msg = "Your order is being prepared."
    elif status == "Cancelled":
        msg = "Your order has been cancelled."
    elif status == "Refunded":
        msg = "Your refund has been processed."
    else:
        msg = f"Status '{status}' — please contact support."
    print(f"{status:12s} → {msg}")



Delivered    → Your order has been delivered.
In Transit   → Your order is on its way.
Cancelled    → Your order has been cancelled.
Refunded     → Your refund has been processed.
Unknown      → Status 'Unknown' — please contact support.


---


## 2 — for Loops

A `for` loop runs a block of code **once for each item** in a collection.

```python
for item in collection:
    do something with item
```

| | `for` | `while` |
|-|-------|---------|
| Use when | You have a known collection | You don't know how many iterations |
| Ends when | Collection is exhausted | Condition becomes False |


In [1]:
# Basic for loop
fruits = ["apple", "banana", "cherry"]

for fruit in fruits:
    print(fruit)


apple
banana
cherry


In [31]:
index,value = (0, 'apple')
print(index,value)

0 apple


In [32]:
# enumerate() — gives index AND value together
# Without: for fruit in fruits      → value only
# With:    for i, fruit in enumerate → index + value

for index,value in enumerate(fruits):
    print(index,value)


0 apple
1 banana
2 cherry


### Accumulation Pattern

Start with an empty list → loop → append one result per iteration.  
This pattern appears in almost every LLM pipeline.

```python
results = []                # start empty
for item in collection:
    results.append(...)     # add one result per iteration
```


In [8]:
test_queries = [
    "Where is my order #3042?",
    "I want to cancel my purchase",
    "Do you have a discount code?",
    "What are the specs of the Classic Monitor?",
]

results = []   # accumulation pattern

for i, query in enumerate(test_queries):
    print(f"iter:{i}, query:{query}")

    q = query.lower()
    if   "cancel" in q or "refund"   in q: agent = "human_agent"
    elif "where"  in q or "track"    in q: agent = "order_agent"
    elif "discount" in q or "price"  in q: agent = "promotions_agent"
    elif "spec"   in q or "product"  in q: agent = "catalog_agent"
    else:                                  agent = "general_agent"

    results.append({"index": i, "query": query, "agent": agent})
    print(results)
    print("#######")




iter:0, query:Where is my order #3042?
[{'index': 0, 'query': 'Where is my order #3042?', 'agent': 'order_agent'}]
#######
iter:1, query:I want to cancel my purchase
[{'index': 0, 'query': 'Where is my order #3042?', 'agent': 'order_agent'}, {'index': 1, 'query': 'I want to cancel my purchase', 'agent': 'human_agent'}]
#######
iter:2, query:Do you have a discount code?
[{'index': 0, 'query': 'Where is my order #3042?', 'agent': 'order_agent'}, {'index': 1, 'query': 'I want to cancel my purchase', 'agent': 'human_agent'}, {'index': 2, 'query': 'Do you have a discount code?', 'agent': 'promotions_agent'}]
#######
iter:3, query:What are the specs of the Classic Monitor?
[{'index': 0, 'query': 'Where is my order #3042?', 'agent': 'order_agent'}, {'index': 1, 'query': 'I want to cancel my purchase', 'agent': 'human_agent'}, {'index': 2, 'query': 'Do you have a discount code?', 'agent': 'promotions_agent'}, {'index': 3, 'query': 'What are the specs of the Classic Monitor?', 'agent': 'catalog

In [9]:
for r in results:
    print(f"[{r['index']}] {r['query'][:38]:38s} → {r['agent']}")

[0] Where is my order #3042?               → order_agent
[1] I want to cancel my purchase           → human_agent
[2] Do you have a discount code?           → promotions_agent
[3] What are the specs of the Classic Moni → catalog_agent


In [7]:
result = []
print(result)
result.append(1)
print(result)
result.append(2)
print(result)
result.append({"name":"prudhvi","age":20,"salary":500})
print(result)

[]
[1]
[1, 2]
[1, 2, {'name': 'prudhvi', 'age': 20, 'salary': 500}]


In [14]:
# Loop over a list of dicts — the database row pattern
# Each dict = one row, keys = column names
reviews = [
    {"review_id": 5001, "rating": 3, "title": "Its fine"},
    {"review_id": 5002, "rating": 5, "title": "Excellent!"},
    {"review_id": 5003, "rating": 4, "title": "Really good"},
    {"review_id": 5004, "rating": 2, "title": "Disappointed"},
]

"★★★☆☆"

for review in reviews:
    stars = "★" * review["rating"] + "☆" * (5 - review["rating"])
    print(f"ID:{review['review_id']}  {stars}  {review['title']}")


ID:5001  ★★★☆☆  Its fine
ID:5002  ★★★★★  Excellent!
ID:5003  ★★★★☆  Really good
ID:5004  ★★☆☆☆  Disappointed


In [13]:
aString = "★" * 3 + "☆" * 2
print(aString)

★★★☆☆


---


## 3 — range()

`range()` generates a sequence of integers without storing them all in memory.

| Form | Produces |
|------|----------|
| `range(stop)` | 0, 1, 2, ..., stop-1 |
| `range(start, stop)` | start, start+1, ..., stop-1 |
| `range(start, stop, step)` | every step-th value from start |


In [39]:
print(list(range(0, 11, 2)))    # [0, 2, 4, 6, 8]
print(list(range(10, 0, -2)))   # [10, 9, 8, ... 1]

[0, 2, 4, 6, 8, 10]
[10, 8, 6, 4, 2]


In [28]:
print(list(range(5)))           # [0, 1, 2, 3, 4]
print(list(range(1, 6)))        # [1, 2, 3, 4, 5]
print(list(range(0, 10, 1)))    # [0, 2, 4, 6, 8]
print(list(range(10, 0, -1)))   # [10, 9, 8, ... 1]  — count down -- descending order

# Batch processing — split 9 queries into batches of 3
total_queries = 9
batch_size    = 3
result = total_queries // batch_size
print(result)
batches = range(result)
print(batches)

for batch_num in batches:
    start = batch_num * batch_size
    end   = start + batch_size - 1
    print(f"Batch {batch_num + 1}: queries {start} to {end}")


[0, 1, 2, 3, 4]
[1, 2, 3, 4, 5]
[0, 2, 4, 6, 8]
[10, 9, 8, 7, 6, 5, 4, 3, 2, 1]
3
range(0, 3)
Batch 1: queries 0 to 2
Batch 2: queries 3 to 5
Batch 3: queries 6 to 8


In [18]:
list(range(1,10))

[1, 2, 3, 4, 5, 6, 7, 8, 9]

In [19]:
print(list(range(0, 10, 2)))

[0, 2, 4, 6, 8]


In [23]:
print(list(range(10, -1, -1)))

[10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0]


---


## 4 — while Loops

A `while` loop runs **as long as its condition is True**.  
Use it when you don't know in advance how many iterations you need.

```python
while condition:
    do something
    update the condition   # REQUIRED — forget this = infinite loop
```

> **Critical rule:** always update whatever the condition checks inside the loop body.


In [ ]:
# Basic while — count from 0 to 4
count = 0

while count < 5:
    # Iter 1 count = 0, 5 -- 0 < 5 -- True
    # Iter 2 count = 1, 5 -- 1 < 5 -- True
    # Iter 3 count = 2, 5 -- 2 < 5 -- True
    # Iter 4 count = 3, 5 -- 3 < 5 -- True
    # Iter 5 count = 4, 5 -- 4 < 5 -- True
    # Iter 5 count = 5, 5 -- 5 < 5 -- False
    print(f"count = {count}")
    count = count + 1    # update the condition — without this: infinite loop

print(f"Loop ended. Final count = {count}")


---


## 5 — break and continue

| Keyword | Effect | Mental model |
|---------|--------|--------------|
| `break` | Exit the loop entirely right now | "I am done — stop the loop" |
| `continue` | Skip this iteration, go to next | "Not this one — move on" |


In [ ]:
# continue — skip even numbers
odds = []
for n in range(1, 10):
    if n % 2 == 0:
        continue        # even → skip this iteration
    odds.append(n)
print("Odd numbers:", odds)

# break — stop at first number above 5
for n in range(1, 10):
    if n > 5:
        print(f"Stopped at {n}")
        break


### ShopSmart — Auto-select Few-shot Examples

In Module 00 Technique 02 you chose few-shot examples by hand.  
Here Python selects them automatically:
- `continue` → skip reviews with rating < 4
- `break` → stop once we have 3 examples collected


In [29]:
aList = [1,2,3,4,5,6]
number_items = len(aList)
print(number_items)

6


In [ ]:
all_reviews = [
    {"review_id": 5001, "rating": 3, "title": "Its fine"},
    {"review_id": 5002, "rating": 5, "title": "Excellent!"},
    {"review_id": 5003, "rating": 4, "title": "Really good"},
    {"review_id": 5004, "rating": 2, "title": "Disappointed"},
    {"review_id": 5005, "rating": 5, "title": "Perfect product"},
    {"review_id": 5006, "rating": 4, "title": "Solid choice"},
]

good_reviews   = []   # accumulator — starts empty
max_to_collect = 3

# ─────────────────────────────────────────────────────────────
# ITERATION TRACE
#
# Iter 1 → review_id=5001  rating=3
#          rating < 4 ?  3 < 4 = True  → continue  (skip, go to Iter 2)
#          good_reviews = []
#
# Iter 2 → review_id=5002  rating=5
#          rating < 4 ?  5 < 4 = False → do NOT skip
#          good_reviews.append(5002)
#          good_reviews = [{5002, 5, 'Excellent!'}]
#          len(good_reviews) >= 3 ?  1 >= 3 = False → keep looping
#
# Iter 3 → review_id=5003  rating=4
#          rating < 4 ?  4 < 4 = False → do NOT skip
#          good_reviews.append(5003)
#          good_reviews = [{5002}, {5003}]
#          len(good_reviews) >= 3 ?  2 >= 3 = False → keep looping
#
# Iter 4 → review_id=5004  rating=2
#          rating < 4 ?  2 < 4 = True  → continue  (skip, go to Iter 5)
#          good_reviews = [{5002}, {5003}]  (unchanged)
#
# Iter 5 → review_id=5005  rating=5
#          rating < 4 ?  5 < 4 = False → do NOT skip
#          good_reviews.append(5005)
#          good_reviews = [{5002}, {5003}, {5005}]
#          len(good_reviews) >= 3 ?  3 >= 3 = True  → BREAK
#          review_id=5006 is never reached
# ─────────────────────────────────────────────────────────────

for review in all_reviews:

    if review["rating"] < 4:
        continue                     # rating too low — skip to next iteration

    good_reviews.append(review)      # rating >= 4 — keep it

    if len(good_reviews) >= max_to_collect:
        break                        # collected enough — exit loop

# Print results
for r in good_reviews:
    print(f"  {r['review_id']}  {'★' * r['rating']}  {r['title']}")

print()
print("Selected titles:", [r["title"] for r in good_reviews])
print(f"review_id=5006 '{all_reviews[5]['title']}' was never reached — loop broke at iter 5")


---


## 6 — List Comprehension

A list comprehension is the **accumulation pattern rewritten in one line**.

```python
# Standard loop (accumulation):
results = []
for item in collection:
    if condition:
        results.append(transform(item))

# List comprehension — same result:
results = [transform(item)  for item in collection  if condition]
#          ↑ WHAT            ↑ THE LOOP              ↑ FILTER (optional)
```

| Use | When |
|-----|------|
| List comprehension | Simple transform or filter on one collection |
| Regular for loop | Complex logic, multiple steps, side effects |


In [ ]:
# Simple examples
squares  = [n * n       for n in range(1, 6)]
evens    = [n           for n in range(1, 11)  if n % 2 == 0]
upper    = [w.upper()   for w in ["hello", "world", "python"]]

print("Squares:", squares)
print("Evens:  ", evens)
print("Upper:  ", upper)


In [ ]:
products = [
    {"name": "Classic Monitor",   "price": 205.21, "stock": 238},
    {"name": "Ultimate Perfume",  "price": 568.17, "stock": 10},
    {"name": "Budget Headphones", "price": 29.99,  "stock": 0},
    {"name": "Yoga Mat Pro",      "price": 45.00,  "stock": 150},
    {"name": "Luxury Cream",      "price": 89.00,  "stock": 0},
]

# Filter — in-stock only
in_stock = [p for p in products  if p["stock"] > 0]
print("In stock:", [p["name"] for p in in_stock])

# Transform — extract one field
names = [p["name"] for p in products]
print("All names:", names)

# Filter + transform — affordable in-stock → formatted prompt lines
prompt_lines = [
    f"- {p['name']} (${p['price']:.2f})"
    for p in products
    if p["stock"] > 0 and p["price"] < 300
]
print("Prompt lines:")
for line in prompt_lines:
    print(" ", line)


In [ ]:
# High-rated review titles — for few-shot examples
high_rated_titles = [
    r["title"]
    for r in all_reviews
    if r["rating"] >= 4
]
print("Few-shot candidates:", high_rated_titles)


---


## 7 — Dict Comprehension

A dict comprehension builds a `{key: value}` dict from a loop in one line.

```python
{key_expr: value_expr  for item in collection  if condition}
```

**Primary use:** turn a list into a fast O(1) lookup dict — instead of scanning the list every time (O(n)).


In [ ]:
# Simple examples
squares_dict = {n: n*n        for n in range(1, 6)}
word_lengths  = {w: len(w)    for w in ["apple", "banana", "cherry"]}

print(squares_dict)            # {1:1, 2:4, 3:9, 4:16, 5:25}
print(squares_dict[3])         # 9 — instant lookup
print(word_lengths)            # {'apple': 5, 'banana': 6, 'cherry': 6}


In [ ]:
# Product lookup index — O(1) access by name
# Without this: scan the list every time → O(n)
# With this:    jump directly to the product → O(1)
product_index = {p["name"]: p for p in products}

classic = product_index["Classic Monitor"]
print(f"Price: {classic['price']}  Stock: {classic['stock']}")

# Review ratings by ID
ratings_by_id = {r["review_id"]: r["rating"] for r in all_reviews}
print(ratings_by_id)
print(f"Review 5002 rating: {ratings_by_id[5002]}")


In [ ]:
# Routing results → lookup dict
routing_index = {r["query"]: r["agent"] for r in results}

for query, agent in list(routing_index.items())[:3]:
    print(f"{query[:40]:40s} → {agent}")


---


## Summary — Day 02

| Concept | Key rule / syntax |
|---------|------------------|
| `if/elif/else` | Most urgent condition first. Always write `else`. Colon required. |
| Comparison ops | `== != > < >= <=` — produce `True` or `False` |
| `in` operator | `"cancel" in q` — substring or membership check |
| `and or not` | Combine conditions. `.lower()` before string matching. |
| `for` loop | `for item in collection:` — known collection |
| `enumerate()` | `for i, item in enumerate(col):` — index + value |
| Accumulation | `results = []` then `results.append(x)` inside loop |
| `range()` | `range(n)` · `range(start,stop)` · `range(start,stop,step)` |
| `while` loop | Unknown iterations. **Always update the condition variable.** |
| Exponential backoff | `wait = 0.1 * (2 ** attempt)` — standard LLM retry |
| `break` | Exit loop entirely. Used: enough collected / API succeeded. |
| `continue` | Skip this iteration. Used: filter out unwanted items. |
| List comprehension | `[expr for item in col if cond]` — filter or transform |
| Dict comprehension | `{key: val for item in col}` — build O(1) lookup index |

**Next:** Day 03 — Functions and Data Structures  
`def`, `*args`, `**kwargs`, all list/dict/set/tuple methods, `sorted()`, `map()`, `filter()`
